# Sesión 4 · Preprocesamiento de datos - EJERCICIOS
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

**Objetivo:** Practicar limpieza de datos completando el código con pistas.

⚠️ **Instrucciones:**
1. Lee cada pista cuidadosamente
2. Completa el código en la celda vacía
3. Ejecuta con `Shift + Enter`
4. Si no funciona, compara con la solución en `sesion-04-preprocesamiento-de-datos.ipynb`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
pd.set_option('display.max_columns', 20)
print('✓ Librerías cargadas')

## EJERCICIO 1: Cargar datos desde GitHub

**Pista:**
- Usa `pd.read_csv(url)` para cargar datos
- La URL está en comments
- Verifica el shape del DataFrame

In [ ]:
# TODO: Completa este código
url = 'https://raw.githubusercontent.com/anclopezg/curso-analisis-datos-energias-renovables/main/docs/datasets/datos_estacion.csv'
df = pd.read_csv(url)

print(f'Dataset: {df.shape}')
print(df.head())

In [ ]:
# Visualización de los 6 gráficos
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# 1. Irradiancia: línea + NaN marcados
ax = axes[0, 0]
ax.plot(df['irradiancia_wm2'].values, color='blue', alpha=0.6, linewidth=0.8)
ax.scatter(df[df['irradiancia_wm2'].isna()].index, [0]*df['irradiancia_wm2'].isna().sum(), color='red', s=30, label='NaN')
ax.set_title('Irradiancia: hay huecos (NaN)', fontweight='bold', color='darkred')
ax.set_ylabel('W/m²')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Temperatura: puntos normales + cruces rojas en -999
ax = axes[0, 1]
prob = df['temperatura_c'] == -999
ax.scatter(df[~prob].index, df[~prob]['temperatura_c'], color='green', s=5, alpha=0.6, label='Normal')
ax.scatter(df[prob].index, [-999]*prob.sum(), color='red', s=50, marker='X', label='Centinelas')
ax.set_title('Temperatura: hay centinelas (-999)', fontweight='bold', color='darkred')
ax.set_ylabel('°C')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Viento: puntos normales + atípicos
ax = axes[0, 2]
atp = df['velocidad_viento_kmh'] > 100
ax.scatter(df[~atp].index, df[~atp]['velocidad_viento_kmh'], color='blue', s=5, alpha=0.6)
ax.scatter(df[atp].index, df[atp]['velocidad_viento_kmh'], color='red', s=50, marker='x', linewidth=2)
ax.set_title('Viento: hay atípicos (×10)', fontweight='bold', color='darkred')
ax.set_ylabel('km/h')
ax.grid(True, alpha=0.3)

# 4. Presión: extremos
ax = axes[1, 0]
ext = df['presion_hpa'] > 1100
ax.scatter(df[~ext].index, df[~ext]['presion_hpa'], color='purple', s=5, alpha=0.6)
ax.scatter(df[ext].index, df[ext]['presion_hpa'], color='red', s=50, marker='D')
ax.set_title('Presión: hay extremos', fontweight='bold', color='darkred')
ax.set_ylabel('hPa')
ax.grid(True, alpha=0.3)

# 5. Duplicados: antes vs después
ax = axes[1, 1]
dup = df.duplicated(keep=False)
ax.bar(['Únicos', 'Duplicados'], [len(df[~dup]), len(df[dup])], color=['green', 'red'])
ax.set_title('Duplicados', fontweight='bold', color='darkred')
ax.set_ylabel('Conteo')

# 6. Resumen de todos los problemas
ax = axes[1, 2]
problemas = {
    'Faltantes': df['irradiancia_wm2'].isna().sum(),
    'Centinelas': (df['temperatura_c'] == -999).sum(),
    'Atípicos': atp.sum(),
    'Duplicados': dup.sum()
}
ax.barh(list(problemas.keys()), list(problemas.values()), color='coral')
ax.set_title('Resumen de problemas', fontweight='bold')

plt.tight_layout()
plt.show()
print('✓ Visualización completa')

### Visualiza los 6 problemas principales

**Pista:** Crea 6 subgráficos (2 filas × 3 columnas) donde cada uno destaca un tipo de problema:
1. **Irradiancia**: línea + puntos rojos en faltantes
2. **Temperatura**: scatter normal + cruces rojas en centinelas (-999)
3. **Viento**: scatter normal + triángulos rojos en atípicos (>100)
4. **Presión**: detecta extremos >1100 hPa
5. **Duplicados**: comparación antes/después
6. **Resumen**: barras horizontales con conteo de problemas

**Materiales:**
- `ax.plot()` para líneas
- `ax.scatter()` para puntos
- `ax.bar()` para barras
- `ax.barh()` para barras horizontales
- Parámetros: `color`, `s` (tamaño), `marker`, `label`

In [ ]:
# TODO: Introduce problemas en el dataset para practicar
df_limpio = df.copy()  # Guarda una copia limpia para después
rng = np.random.default_rng(42)

# Faltantes: 50 valores NaN aleatorios en irradiancia
indices_nan = rng.choice(len(df), 50, replace=False)
df.loc[indices_nan, 'irradiancia_wm2'] = np.nan

# Centinelas: 30 valores imposibles (-999) en temperatura
indices_cent = rng.choice(len(df), 30, replace=False)
df.loc[indices_cent, 'temperatura_c'] = -999

# Atípicos: 8 valores multiplicados por 10 en velocidad de viento
indices_atp = rng.choice(len(df), 8, replace=False)
df.loc[indices_atp, 'velocidad_viento_kmh'] *= 10

# Duplicados: repite 10 filas aleatorias
indices_dup = rng.choice(len(df), 10, replace=False)
df = pd.concat([df, df.iloc[indices_dup]], ignore_index=True)

print('✓ Problemas introducidos:')
print(f'  - Faltantes: {indices_nan.size} NaN en irradiancia')
print(f'  - Centinelas: {indices_cent.size} valores -999 en temperatura')
print(f'  - Atípicos: {indices_atp.size} valores ×10 en viento')
print(f'  - Duplicados: {len(indices_dup)} filas repetidas')
print(f'Dataset ahora tiene {len(df)} filas (antes {len(df_limpio)})')

## EJERCICIO EXPLORATORIO: Visualizar todos los problemas

Antes de limpiar nada, veamos **dónde están los problemas**. Con un solo gráfico puedes detectar:
- **Faltantes** (NaN): huecos en la línea
- **Centinelas** (-999, valores imposibles): picos raros
- **Atípicos** (extremos): valores 10× más grandes
- **Duplicados**: filas repetidas

**Concepto:** La visualización es el primer paso del análisis. Mira primero, pregunta después, limpia con propósito.

## EJERCICIO 2: Introducir problemas

**Pista:**
- Crea una copia: `df = df.copy()`
- Usa `rng = np.random.default_rng(42)` para reproducibilidad
- Asigna NaN: `df.loc[índices, 'columna'] = np.nan`
- Crea centinelas: `df.loc[índices, 'columna'] = -999`

In [ ]:
# TODO: Completa este código
df_limpio = df.copy()
df = ...
rng = ...

# Agregar 50 faltantes a irradiancia
indices_nan = ...
df.loc[indices_nan, 'irradiancia_wm2'] = ...

# Agregar 30 centinelas a temperatura
indices_centinela = ...
df.loc[indices_centinela, 'temperatura_c'] = ...

print('✓ Problemas introducidos')

## EJERCICIO 3: Visualizar un problema

**Pista:**
- Crea un figura: `fig, ax = plt.subplots()`
- Plot línea: `ax.plot(df['columna'].values, color='blue')`
- Scatter puntos faltantes: `ax.scatter(df[df['col'].isna()].index, ...)`
- Añade título y leyenda

In [ ]:
# TODO: Crea un gráfico de irradiancia mostrando dónde están los NaN
fig, ax = ...

# Línea de irradiancia
ax.plot(...)

# Scatter de puntos faltantes (color rojo, s=30)
ax.scatter(...)

# Configura
ax.set_title('Irradiancia: dónde están los huecos')
ax.set_ylabel('W/m²')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## EJERCICIO 4: Eliminar duplicados

**Pista:**
- Cuenta: `df.duplicated().sum()`
- Elimina: `df.drop_duplicates()`
- Reinicia índice: `.reset_index(drop=True)`

In [ ]:
# TODO: Completa
print(f'Duplicados ANTES: {df.duplicated().sum()}')

df = ...

print(f'Duplicados DESPUÉS: {df.duplicated().sum()}')
print('✓ Duplicados eliminados')

## EJERCICIO 5: Detectar atípicos con IQR

**Pista:**
1. `q1 = df['columna'].quantile(0.25)`
2. `q3 = df['columna'].quantile(0.75)`
3. `iqr = q3 - q1`
4. `limite = q3 + 1.5 * iqr`
5. `atipicos = df['columna'] > limite`

In [ ]:
# TODO: Calcula Q1, Q3, IQR y límite para velocidad_viento_kmh
col = 'velocidad_viento_kmh'
q1 = ...
q3 = ...
iqr = ...
lim_sup = ...

# Detecta atípicos
atp = ...

print(f'Q1: {q1:.2f}, Q3: {q3:.2f}, IQR: {iqr:.2f}')
print(f'Límite: {lim_sup:.2f}')
print(f'Atípicos: {atp.sum()}')

## EJERCICIO 6: Convertir tipos de datos

**Pista:**
- Fecha: `pd.to_datetime(df['fecha_hora'], format='mixed', errors='coerce')`
- Centinela a NaN: `.replace(-999, np.nan)`

In [ ]:
# TODO: Completa
df['fecha_hora'] = ...
df['temperatura_c'] = ...

print('✓ Tipos convertidos')
print(df.dtypes)

## EJERCICIO 7: Interpolar faltantes

**Pista:**
1. Ordena por fecha: `df.sort_values('fecha_hora')`
2. Selecciona columnas numéricas: `df.select_dtypes(include=[np.number]).columns`
3. Interpola: `.interpolate(method='linear').bfill().ffill()`

In [ ]:
# TODO: Completa
print(f'Faltantes ANTES: {df.isna().sum().sum()}')

df = ...
cols_num = ...
df[cols_num] = ...

print(f'Faltantes DESPUÉS: {df.isna().sum().sum()}')
print('✓ Interpolación completada')

## EJERCICIO 8: Crear variables nuevas

**Pista:**
- Extrae hora: `df['fecha_hora'].dt.hour`
- Extrae día semana: `df['fecha_hora'].dt.dayofweek` (0=lunes, 6=domingo)
- Crea booleano: `((df['hora'] < 6) | (df['hora'] >= 20)).astype(int)`

In [ ]:
# TODO: Crea hora, dia_semana, es_noche
df['hora'] = ...
df['dia_semana'] = ...
df['es_noche'] = ...

print('✓ Variables nuevas creadas')
print(df[['fecha_hora', 'hora', 'dia_semana', 'es_noche']].head())

## EJERCICIO 9: División train/test

**Pista:**
1. Set index: `df.set_index('fecha_hora')`
2. Calcula corte: `corte = int(len(df) * 0.8)`
3. Divide: `X_train = X.iloc[:corte]`

In [ ]:
# TODO: Completa
df = ...
corte = ...

features = ['temperatura_c', 'irradiancia_wm2', 'velocidad_viento_kmh', 'humedad_relativa', 'presion_hpa', 'hora', 'dia_semana']
X = ...
y = ...

X_train, X_test = ...
y_train, y_test = ...

print(f'Entrenamiento: {X_train.shape}')
print(f'Prueba: {X_test.shape}')

## EJERCICIO 10: Escalado

**Pista:**
1. Crea scaler: `StandardScaler().fit(X_train[cols])`
2. Transforma: `scaler.transform(X_train[cols])`
3. Copia y asigna: `X_train_s[cols] = ...`

In [ ]:
# TODO: Completa
cols_scale = ['temperatura_c', 'irradiancia_wm2', 'velocidad_viento_kmh', 'humedad_relativa', 'presion_hpa']

scaler = ...
X_train_s = ...
X_test_s = ...

X_train_s[cols_scale] = ...
X_test_s[cols_scale] = ...

print('✓ Escalado completado')
print(X_train_s[cols_scale].describe().round(2))

## RETO FINAL

**Ejercicio bonus:**
1. Crea un gráfico que compare la temperatura ANTES y DESPUÉS de limpiar
2. Usa dos subplots (antes/después)
3. Destaca en rojo los valores problemáticos (centinelas en ANTES, NaN en DESPUÉS)

In [ ]:
# TODO: Crea tu propio gráfico de antes/después para temperatura
# Pista: usa df_limpio (limpio) vs df (sucio)

# Aquí va tu código

plt.show()

## Soluciones

Si necesitas ver la solución completa, abre:
- `../sesion-04-preprocesamiento-de-datos.ipynb` (solución con gráficos)

---

**Checklist:**
- ✓ Cargar datos desde URL
- ✓ Introducir problemas realistas
- ✓ Visualizar problemas
- ✓ Eliminar duplicados
- ✓ Detectar atípicos (IQR)
- ✓ Convertir tipos
- ✓ Interpolar faltantes
- ✓ Variables nuevas
- ✓ División train/test
- ✓ Escalado

¡Excelente trabajo! 🎉